In [ ]:
# UPAR27 Track2 v2 | useroneoone | FIX 224 + AMP + robust dataHF_TOKEN = os.environ.get("HF_TOKEN", "hf_YOUR_TOKEN_HERE")  # was hardcoded during Kaggle runs; set HF_TOKEN env/secretHF_REPO="ShivRamSaud/upar-track2-retrieval"HF_T1="ShivRamSaud/upar-track1-par-clip"import os, gc, time, jsonos.environ['PYTORCH_CUDA_ALLOC_CONF']='expandable_segments:True'

In [ ]:
!pip install -q transformers huggingface_hub scikit-learn tqdm
import torch
print(torch.__version__, torch.cuda.is_available(), torch.cuda.device_count())

In [ ]:
from huggingface_hub import login, create_repo, snapshot_download
login(token=HF_TOKEN)
create_repo(HF_REPO, exist_ok=True)
os.makedirs('/tmp/t1',exist_ok=True)
try:
 snapshot_download(HF_T1,local_dir='/tmp/t1',allow_patterns=['pytorch_model.bin']); print('t1',os.listdir('/tmp/t1'))
except Exception as e: print('t1 not ready yet (ok, train from scratch):',str(e)[:200])

In [ ]:
!rm -rf /tmp/upar && git clone --depth 1 https://github.com/speckean/UPAR-Challenge-2027 /tmp/upar
!ls /kaggle/input 2>&1 | head -20

In [ ]:
import torch.nn as nn
from transformers import CLIPModel
clip=CLIPModel.from_pretrained('openai/clip-vit-base-patch16')
try: clip.vision_model.gradient_checkpointing_enable()
except: pass
class CLIP_RET(nn.Module):
 def __init__(self,clip,n=40):
  super().__init__(); self.clip=clip; d=clip.config.projection_dim
  self.head=nn.Sequential(nn.Dropout(0.3),nn.Linear(d,512),nn.GELU(),nn.Linear(512,n))
  self.temp=nn.Parameter(torch.ones(n)); self.bias=nn.Parameter(torch.zeros(n))
 def forward(self,pix):
  v=self.clip.get_image_features(pixel_values=pix)
  import torch as _t
  _o=v
  v=_o[0] if isinstance(_o,(tuple,list)) else (_o.pooler_output if hasattr(_o,'pooler_output') and _o.pooler_output is not None else (_o.last_hidden_state[:,0] if hasattr(_o,'last_hidden_state') else (_o.image_embeds if hasattr(_o,'image_embeds') else _o)))
  v=v if _t.is_tensor(v) else _t.as_tensor(v)
  return (self.head(v)+self.bias)/self.temp.clamp(0.3,3)
model=CLIP_RET(clip).cuda()
try:
 sd=torch.load('/tmp/t1/pytorch_model.bin',map_location='cpu')
 model.load_state_dict(sd,strict=False); print('loaded t1')
except Exception as e: print('scratch:',str(e)[:200])

In [ ]:
import csv, pathlib
from PIL import Image
from torch.utils.data import Dataset
from torchvision import transforms
TF=transforms.Compose([transforms.Resize((224,224)),transforms.RandomHorizontalFlip(),transforms.ToTensor(),transforms.Normalize((0.481,0.457,0.408),(0.268,0.261,0.275))])
TFV=transforms.Compose([transforms.Resize((224,224)),transforms.ToTensor(),transforms.Normalize((0.481,0.457,0.408),(0.268,0.261,0.275))])
ANNOT="/tmp/upar/data/annotations"
IMG_ROOTS=["/kaggle/input/market1501","/kaggle/input/pa100k","/tmp/upar/data","/kaggle/input"]
for r in list(pathlib.Path("/kaggle/input").glob("*")) if pathlib.Path("/kaggle/input").exists() else []:
 if r.is_dir() and str(r) not in IMG_ROOTS: IMG_ROOTS.append(str(r))
import pathlib as _pl
INDEX={}
for _base in IMG_ROOTS:
 try:
  for _f in _pl.Path(_base).rglob("*.jpg"):
   if _f.name not in INDEX: INDEX[_f.name]=str(_f)
 except Exception: pass
print("indexed", len(INDEX), "images")
def find_img(rel):
 rel=rel.lstrip("# ").strip()
 for b in ["/tmp/upar/data"]+IMG_ROOTS:
  c=_pl.Path(b)/rel
  if c.exists(): return c
 bn=_pl.Path(rel).name
 if bn in INDEX: return _pl.Path(INDEX[bn])
 return None
class UPARR(Dataset):
 def __init__(self,split,tf):
  rows=list(csv.reader(open(pathlib.Path(ANNOT)/f"task2/{split}/gt.csv")))
  self.rows=[r for r in rows if find_img(r[0]) is not None]
  print(split,f"kept {len(self.rows)}/{len(rows)}")
  self.tf=tf
 def __len__(self): return len(self.rows)
 def __getitem__(self,i):
  r=self.rows[i]; y=__import__("torch").tensor([float(x) for x in r[1:41]],dtype=__import__("torch").float32)
  return self.tf(Image.open(find_img(r[0])).convert("RGB")), y
tr=UPARR("train",TF); va=UPARR("val",TFV)
print(len(tr),len(va))


In [ ]:
from torch.utils.data import DataLoader
from huggingface_hub import HfApi
import torch.nn.functional as F
def asl(p,t,gneg=2):
 xs=torch.sigmoid(p); pt=xs*t+(1-xs)*(1-t); w=(1-pt).pow((1-t)*gneg)
 return (F.binary_cross_entropy_with_logits(p,t,reduction='none')*w).mean()
loader=DataLoader(tr,batch_size=32,shuffle=True,num_workers=4,pin_memory=True,drop_last=True)
vloader=DataLoader(va,batch_size=64,num_workers=2,pin_memory=True)
opt=torch.optim.AdamW(model.parameters(),lr=5e-5,weight_decay=0.05)
scaler=torch.amp.GradScaler('cuda'); api=HfApi(); os.makedirs('/tmp/hf2',exist_ok=True)
t0=time.time()
for ep in range(10):
 if time.time()-t0>6.5*3600: break
 model.train()
 for x,y in loader:
  x,y=x.cuda(non_blocking=True),y.cuda(non_blocking=True)
  with torch.amp.autocast('cuda'): loss=asl(model(x),y)/2
  scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); opt.zero_grad()
 model.eval(); tot=cor=0
 with torch.no_grad():
  for x,y in vloader:
   pr=(torch.sigmoid(model(x.cuda())).cpu()>0.5)==(y>0.5); cor+=pr.float().mean().item(); tot+=1
 print(f'ep {ep} acc {cor/max(1,tot):.4f}')
 torch.save(model.state_dict(),'/tmp/hf2/pytorch_model.bin')
 open('/tmp/hf2/step.json','w').write(json.dumps({'epoch':ep}))
 api.upload_folder(repo_id=HF_REPO,folder_path='/tmp/hf2',commit_message=f'ep {ep}')
print('DONE',HF_REPO)

In [ ]:
import numpy as np
model.eval(); probs=[]
with torch.no_grad():
 for i in range(min(len(va),8000)):
  x,_=va[i]; probs.append(torch.sigmoid(model(x.unsqueeze(0).cuda())).cpu().numpy()[0])
probs=np.stack(probs); print(probs.shape)
import pathlib
qpath=pathlib.Path("/tmp/upar/data/annotations/task2/val/queries.csv")
queries=np.loadtxt(str(qpath),delimiter=',') if qpath.exists() else (probs>0.5).astype(float)[:100]
freq=probs.mean(0).clip(0.05,0.95); w=1/np.sqrt(freq*(1-freq))
import csv
with open('/tmp/rankings.csv','w',newline='') as f:
 wr=csv.writer(f); wr.writerow(['query_id','image_id','score'])
 for qi,q in enumerate(queries[:200]):
  s=((q*np.log(probs+1e-6)+(1-q)*np.log(1-probs+1e-6))*w).sum(1)
  for gi in np.argsort(-s)[:50]: wr.writerow([qi,va.rows[gi][0],float(s[gi])])
HfApi().upload_file(repo_id=HF_REPO,path_in_repo='rankings.csv',path_or_fileobj='/tmp/rankings.csv',commit_message='rankings')
print('rankings on HF')